# Silver — limpeza, padronização e integração

Objetivo: partir das 6 tabelas Bronze (`uf`, `municipio`, `meta_alfabetizacao_brasil/uf/municipio`, `alunos`),
normalizar o código IBGE do município, decodificar e padronizar a coluna `rede`, integrar tudo num
modelo conformado de 4 tabelas (município, UF, Brasil, alunos) e rodar scripts de qualidade
(duplicidade, nulos, consistência) antes de gravar em `s3://.../silver/`.

In [ ]:
import awswrangler as wr
import pandas as pd
import boto3

BUCKET = "brazil-literacy-lakehouse-joaopaulo"
REGION = "sa-east-1"

s3_client = boto3.client("s3", region_name=REGION)

df_uf = wr.s3.read_parquet(path=f"s3://{BUCKET}/bronze/uf/", dataset=True)
df_municipio = wr.s3.read_parquet(path=f"s3://{BUCKET}/bronze/municipio/", dataset=True)
df_meta_brasil = wr.s3.read_parquet(path=f"s3://{BUCKET}/bronze/meta_alfabetizacao_brasil/", dataset=True)
df_meta_uf = wr.s3.read_parquet(path=f"s3://{BUCKET}/bronze/meta_alfabetizacao_uf/", dataset=True)
df_meta_municipio = wr.s3.read_parquet(path=f"s3://{BUCKET}/bronze/meta_alfabetizacao_municipio/", dataset=True)
df_alunos = wr.s3.read_parquet(path=f"s3://{BUCKET}/bronze/alunos/", dataset=True)

for nome, df in [
    ("uf", df_uf), ("municipio", df_municipio), ("meta_brasil", df_meta_brasil),
    ("meta_uf", df_meta_uf), ("meta_municipio", df_meta_municipio), ("alunos", df_alunos),
]:
    print(f"{nome}: {len(df):,} linhas, {len(df.columns)} colunas")

## Passo 1 — Normalização do código IBGE

Investigação: `id_municipio` já vem consistente em 7 dígitos em todas as tabelas que o têm (sem
padding/correção de tipo necessária). O trabalho real é derivar `sigla_uf` nas 3 tabelas que só
têm `id_municipio` (`municipio`, `meta_alfabetizacao_municipio`, `alunos`) — essa tabela `municipio`
específica do dataset do desafio não vem com `sigla_uf` (diferente da tabela padrão de diretório
do `basedosdados`).

In [ ]:
# Mapeamento oficial IBGE: 2 primeiros dígitos do código do município -> sigla da UF
CODIGO_UF_PARA_SIGLA = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
    "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}


def adicionar_sigla_uf(df: pd.DataFrame) -> pd.DataFrame:
    """Deriva sigla_uf a partir dos 2 primeiros dígitos de id_municipio (vetorizado, rápido mesmo em milhões de linhas)."""
    df = df.copy()
    df["sigla_uf"] = df["id_municipio"].str[:2].map(CODIGO_UF_PARA_SIGLA)
    return df

In [ ]:
df_municipio = adicionar_sigla_uf(df_municipio)
df_meta_municipio = adicionar_sigla_uf(df_meta_municipio)
df_alunos = adicionar_sigla_uf(df_alunos)

for nome, df in [("municipio", df_municipio), ("meta_alfabetizacao_municipio", df_meta_municipio), ("alunos", df_alunos)]:
    nulos = df["sigla_uf"].isna().sum()
    print(f"{nome}: sigla_uf nula em {nulos:,} linhas ({nulos/len(df):.2%})")

## Passo 2 — Decodificação e padronização de `rede`

Investigação revelou:
- `ano`: `uf`/`municipio`/`meta_alfabetizacao_municipio`/`alunos` cobrem 2023-2024; `meta_alfabetizacao_brasil`/`meta_alfabetizacao_uf` cobrem 2023-2025 (meta é projeção futura).
- `serie`: sempre `'2'` em todo lugar (dataset é só sobre o 2º ano) — mantida como documentação, custo zero.
- `rede`: vem como código numérico em `uf`/`municipio`/`alunos`, e como texto nas tabelas de meta — decodificado usando a tabela oficial `dicionario` do próprio dataset (`basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`).

In [ ]:
BILLING_PROJECT_ID = "brazil-literacy-lakehouse"

query = """
SELECT *
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
"""
df_dicionario = bd.read_sql(query, billing_project_id=BILLING_PROJECT_ID)
df_dicionario.shape

In [ ]:
rede_dict_raw = df_dicionario[df_dicionario["nome_coluna"] == "rede"]
REDE_DICT = dict(zip(rede_dict_raw["chave"].astype(str), rede_dict_raw["valor"]))
print(REDE_DICT)


def padronizar_rede_codigo(df):
    """Pra uf/municipio/alunos: troca o código numérico pelo texto oficial do dicionário."""
    df = df.copy()
    df["rede"] = df["rede"].astype(str).map(REDE_DICT)
    return df


def padronizar_rede_publica(df):
    """Pra meta_alfabetizacao_brasil/uf: troca 'Pública' pelo texto canônico da chave 5.

    Decisão: 'Pública' mapeia pra chave 5 (Estadual e Municipal), não a chave 6 (+Federal).
    Motivo: as chaves 1 (Federal) e 6 (Pública c/ Federal) nunca aparecem nos dados reais de
    uf/municipio (só aparecem 0, 2, 3, 5) — evidência de que a rede Federal tem participação
    praticamente nula nessa avaliação (2º ano do fundamental), então a convenção de 'Pública'
    usada pelo próprio INEP/Base dos Dados nos resultados publicados é a chave 5.
    """
    df = df.copy()
    df["rede"] = df["rede"].replace({"Pública": REDE_DICT["5"]})
    return df

In [ ]:
df_uf = padronizar_rede_codigo(df_uf)
df_municipio = padronizar_rede_codigo(df_municipio)
df_alunos = padronizar_rede_codigo(df_alunos)
df_meta_brasil = padronizar_rede_publica(df_meta_brasil)
df_meta_uf = padronizar_rede_publica(df_meta_uf)
# meta_alfabetizacao_municipio já usa 'Municipal', que bate direto com o texto canônico da chave 3

for nome, df in [
    ("uf", df_uf), ("municipio", df_municipio), ("alunos", df_alunos),
    ("meta_brasil", df_meta_brasil), ("meta_uf", df_meta_uf), ("meta_municipio", df_meta_municipio),
]:
    print(f"{nome} -> {sorted(df['rede'].dropna().unique().tolist())}")

### `nivel_alfabetizacao` — achado empírico

Só existe em `meta_alfabetizacao_municipio`, sem entrada no dicionário oficial. Investigação
(`groupby` por nível, olhando `min`/`max`/`mean` de `taxa_alfabetizacao`) mostrou que é uma
classificação em faixas de 10 pontos percentuais, sem sobreposição entre os grupos: nível 0 =
taxa < 40%, nível 1 = 40-50%, nível 2 = 50-60%, nível 3 = 60-70%, nível 4 = 70-80%, nível 5 = ≥80%.
Nulo sempre que `taxa_alfabetizacao` também é nula (classificação derivada). Achado próprio —
não documentado oficialmente pela Base dos Dados.

In [ ]:
print(df_meta_municipio.groupby("nivel_alfabetizacao", observed=True)["taxa_alfabetizacao"].agg(["min", "max", "mean", "count"]))

## Passo 3 — Integração: modelo conformado de 4 tabelas

As 6 fontes têm granularidades diferentes (município, UF, Brasil, aluno individual) — uma tabela
única "flat" geraria duplicação sem sentido (broadcast da linha nacional pra 24 mil municípios) ou
explosão de linhas (aluno × agregado). Modelo adotado: cada granularidade continua como sua
própria tabela Silver, todas com chaves/nomes/tipos consistentes entre si:

1. `silver_municipio` = `municipio` **LEFT JOIN** `meta_alfabetizacao_municipio` em `(id_municipio, ano, rede)`
2. `silver_uf` = `uf` **LEFT JOIN** `meta_alfabetizacao_uf` em `(sigla_uf, ano, rede)`
3. `silver_brasil` = `meta_alfabetizacao_brasil` (já vem com resultado + meta juntos na própria tabela)
4. `silver_alunos` = `alunos` (granularidade separada de propósito)

LEFT JOIN, não INNER — inner descartaria linhas de resultado que não têm meta correspondente
(ex: `municipio` tem 4 categorias de `rede`, mas `meta_alfabetizacao_municipio` só tem `'Municipal'`).

In [ ]:
LIMIAR_DIVERGENCIA_REAL = 1.0  # acima disso não é mais arredondamento (ver achado abaixo)


def montar_silver_resultado(df_resultado, df_meta, chaves, limiar_divergencia=LIMIAR_DIVERGENCIA_REAL):
    """Junta resultado (municipio/uf) com sua respectiva tabela de meta.

    Mantém todas as linhas do resultado (left join) e marca divergência real entre as duas
    fontes de taxa_alfabetizacao, quando ambas existem.

    Achado de qualidade: comparando taxa_alfabetizacao de resultado vs. meta, a maioria das
    diferenças é arredondamento (mediana = 0). Em nível município, 3 casos têm divergência real
    (> 1 ponto, todos em 2023): id_municipio 2304905 (16.05pt), 2305100 (36.07pt), 5106752 (2.06pt).
    Em nível UF, 0 divergências reais (máximo 0.05, tudo arredondamento). Mantido o valor de
    resultado (municipio/uf) como oficial, com a divergência rastreável via flag em vez de escondida.
    """
    df = df_resultado.merge(df_meta, on=chaves, how="left", suffixes=("", "_meta"))
    if "taxa_alfabetizacao_meta" in df.columns:
        diff = (df["taxa_alfabetizacao"] - df["taxa_alfabetizacao_meta"]).abs()
        df["taxa_alfabetizacao_divergente"] = diff > limiar_divergencia
        df = df.drop(columns=["taxa_alfabetizacao_meta"])
    return df


silver_municipio = montar_silver_resultado(df_municipio, df_meta_municipio, ["id_municipio", "ano", "rede"])
silver_uf = montar_silver_resultado(df_uf, df_meta_uf, ["sigla_uf", "ano", "rede"])
silver_brasil = df_meta_brasil.copy()  # já vem com resultado + meta juntos na própria tabela
silver_alunos = df_alunos.copy()

print(f"silver_municipio: {silver_municipio.shape} | divergências reais: {silver_municipio['taxa_alfabetizacao_divergente'].sum()}")
print(f"silver_uf: {silver_uf.shape} | divergências reais: {silver_uf['taxa_alfabetizacao_divergente'].sum()}")
print(f"silver_brasil: {silver_brasil.shape}")
print(f"silver_alunos: {silver_alunos.shape}")

## Passo 4 — Scripts de qualidade (duplicidade, nulos, chaves)

In [ ]:
def checar_qualidade(nome, df, chaves):
    print(f"=== {nome} ({len(df):,} linhas) ===")
    print(f"Duplicadas (linha inteira): {df.duplicated().sum()}")
    print(f"Duplicadas (por chave {chaves}): {df.duplicated(subset=chaves).sum()}")

    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    if len(nulos):
        print("Colunas com nulos:")
        for col, qtd in nulos.items():
            print(f"  {col}: {qtd:,} ({qtd/len(df):.1%})")
    else:
        print("Sem nulos em nenhuma coluna.")
    print()


checar_qualidade("silver_municipio", silver_municipio, ["id_municipio", "ano", "rede"])
checar_qualidade("silver_uf", silver_uf, ["sigla_uf", "ano", "rede"])
checar_qualidade("silver_brasil", silver_brasil, ["ano", "rede"])
checar_qualidade("silver_alunos", silver_alunos, ["id_aluno", "ano"])

### Achados de nulos — todos investigados e explicados como estruturais (nenhum requer imputação)

- `proporcao_aluno_nivel_0..8` (~48% nulo em município/UF): **resolve o achado pendente desde a
  Fase 1.** 2023 = 0% preenchido em qualquer rede, 2024 = 100% preenchido — essa quebra por nível
  de proficiência só passou a ser publicada a partir de 2024.
- `meta_alfabetizacao_*`/`nivel_alfabetizacao`/`percentual_participacao`: só existem pra rede
  `Municipal` (município) / `Pública (Estadual e Municipal)` (UF) — nula em qualquer outra rede por
  definição (96% de preenchimento dentro da rede correta; os ~4% restantes são municípios sem meta
  cadastrada).
- `proficiencia`/`peso_aluno` em `alunos` (13,3% nulo): determinístico — só existe valor quando
  `presenca=1` **e** `preenchimento_caderno=1`.

### Achado de duplicidade: bug de ingestão na Bronze

`silver_brasil` tinha 3 linhas duplicadas (uma pra cada ano, repetida 2x). Causa raiz identificada:
o notebook `02_ingestao_bronze.ipynb` tinha uma célula de teste que gravava `meta_alfabetizacao_brasil`
manualmente antes do loop principal gravar ela de novo — como `wr.s3.to_parquet` não sobrescreve
por padrão, sobraram 2 arquivos Parquet por partição de ano em `bronze/meta_alfabetizacao_brasil/`
(confirmado direto no S3, gerados 6 minutos um do outro). Isolado só nessa tabela — as outras 5
batem exatamente com a contagem de linhas documentada na Fase 2. Corrigido aqui com
`drop_duplicates()`, e a célula de teste foi removida do notebook de ingestão (que agora usa
`mode="overwrite"`, pra esse tipo de bug não se repetir).

In [ ]:
silver_brasil = silver_brasil.drop_duplicates().reset_index(drop=True)
print(f"silver_brasil após dedup: {silver_brasil.shape}")

SILVER_PREFIX = "silver"

wr.s3.to_parquet(df=silver_municipio, path=f"s3://{BUCKET}/{SILVER_PREFIX}/municipio/", dataset=True, partition_cols=["ano"], mode="overwrite")
wr.s3.to_parquet(df=silver_uf, path=f"s3://{BUCKET}/{SILVER_PREFIX}/uf/", dataset=True, partition_cols=["ano"], mode="overwrite")
wr.s3.to_parquet(df=silver_brasil, path=f"s3://{BUCKET}/{SILVER_PREFIX}/brasil/", dataset=True, partition_cols=["ano"], mode="overwrite")
wr.s3.to_parquet(df=silver_alunos, path=f"s3://{BUCKET}/{SILVER_PREFIX}/alunos/", dataset=True, partition_cols=["ano"], mode="overwrite")

print("Silver escrita com sucesso.")

## Validação final — relê do S3 e confere contra os números esperados

In [ ]:
esperado = {
    "municipio": 23_995,
    "uf": 145,
    "brasil": 3,
    "alunos": 3_867_999,
}

for nome, linhas_esperadas in esperado.items():
    df_check = wr.s3.read_parquet(path=f"s3://{BUCKET}/{SILVER_PREFIX}/{nome}/", dataset=True)
    status = "OK" if len(df_check) == linhas_esperadas else "DIVERGENTE"
    print(f"{nome}: {len(df_check):,} linhas (esperado {linhas_esperadas:,}) -> {status}")
    print(f"  duplicadas: {df_check.duplicated().sum()}")

df_municipio_check = wr.s3.read_parquet(path=f"s3://{BUCKET}/{SILVER_PREFIX}/municipio/", dataset=True)
print(f"\nsilver_municipio -> divergências: {df_municipio_check['taxa_alfabetizacao_divergente'].sum()} (esperado 3)")